In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

o:\Hackthons\KrishiOS\ai


In [2]:
import json
import time

import torch

from configs.config import *

from training.model import build_model
print("✅ Libraries Loaded")

✅ Libraries Loaded


In [3]:
model = build_model(NUM_CLASSES)

checkpoint = torch.load(
    PROJECT_ROOT / "models" / "best_model.pth",
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(DEVICE)

model.eval()

print("✅ Model Loaded")

✅ Model Loaded


In [4]:
dummy_input = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
).to(DEVICE)

print(dummy_input.shape)

torch.Size([1, 3, 224, 224])


In [5]:
torchscript_model = torch.jit.trace(
    model,
    dummy_input,
)

export_path = PROJECT_ROOT / "models" / "model.ts"

torchscript_model.save(export_path)

print("✅ TorchScript model exported")

print(export_path)

✅ TorchScript model exported
O:\Hackthons\KrishiOS\ai\models\model.ts


In [7]:
loaded_model = torch.jit.load(
    export_path,
    map_location=DEVICE,
)

loaded_model.eval()

print("✅ Exported model loaded")

✅ Exported model loaded


In [8]:
with torch.no_grad():

    original_output = model(dummy_input)

    exported_output = loaded_model(dummy_input)

difference = torch.max(
    torch.abs(original_output - exported_output)
)

print(f"Maximum Difference: {difference:.8f}")

Maximum Difference: 0.00000000


In [9]:
runs = 100

start = time.perf_counter()

for _ in range(runs):

    with torch.no_grad():

        loaded_model(dummy_input)

end = time.perf_counter()

avg_time = (end - start) / runs

print(f"Average Inference Time: {avg_time*1000:.2f} ms")

Average Inference Time: 17.30 ms


In [10]:
report = {
    "model": "EfficientNet-B0",
    "classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "device": str(DEVICE),
    "average_inference_ms": avg_time * 1000,
    "max_difference": float(difference),
    "export_format": "TorchScript",
}

report_path = PROJECT_ROOT / "outputs" / "export_report.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=4)

print("✅ Export report saved")
print(report_path)

✅ Export report saved
O:\Hackthons\KrishiOS\ai\outputs\export_report.json


In [11]:
print("=" * 60)
print("KrishiOS AI Export Complete")
print()

print("Artifacts Generated")
print("-------------------")
print(f"Model        : {export_path}")
print(f"Report       : {report_path}")
print()
print(f"Inference    : {avg_time * 1000:.2f} ms")
print(f"Difference   : {difference:.8f}")

KrishiOS AI Export Complete

Artifacts Generated
-------------------
Model        : O:\Hackthons\KrishiOS\ai\models\model.ts
Report       : O:\Hackthons\KrishiOS\ai\outputs\export_report.json

Inference    : 17.30 ms
Difference   : 0.00000000
